# Cattle Weight Estimation — Step 1: Complete Dataset Exploration

**Goal:** Produce a full scientific audit of the dataset before writing a single model.

Covers:
- Folder structure & file inventory
- Image resolution, quality, duplicates, corruption
- Weight / age / sex label distributions
- Mask quality & area
- Keypoint quality
- Camera viewpoint classification
- Correlation matrix (body features vs weight)
- Outlier detection
- MAPE achievability analysis
- Feature engineering candidates

In [ ]:
# ── Install / upgrade once ────────────────────────────────────────────────────
import subprocess, sys
def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

pip('Pillow', 'opencv-python-headless', 'scikit-image', 'imagehash',
    'pandas', 'numpy', 'matplotlib', 'seaborn', 'scipy',
    'tqdm', 'pycocotools', 'albumentations')

In [ ]:
import os, json, glob, hashlib, warnings, re
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image, ImageFilter
import cv2
import imagehash
from tqdm.auto import tqdm
from scipy import stats
from skimage import measure as sk_measure
warnings.filterwarnings('ignore')

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}

# ── Print everything mounted so we can debug path issues ──────────────────────
INPUT = Path('/kaggle/input')
print('=== /kaggle/input contents ===')
for entry in sorted(INPUT.iterdir()):
    try:
        n_files = sum(1 for _ in entry.rglob('*') if _.is_file())
        print(f'  {entry.name}/  ({n_files} files)')
    except Exception as e:
        print(f'  {entry.name}/  (error: {e})')

def _count_images(p):
    try:
        return sum(1 for f in p.rglob('*') if f.is_file() and f.suffix.lower() in IMG_EXTS)
    except:
        return 0

# Try the expected slug first; fall back to the dir with most images
EXPECTED = INPUT / 'cattle-weight-detection-model-dataset-12k'
if EXPECTED.exists() and _count_images(EXPECTED) > 0:
    ROOT = EXPECTED
else:
    candidates = sorted(
        [(d, _count_images(d)) for d in INPUT.iterdir() if d.is_dir()],
        key=lambda x: x[1], reverse=True
    )
    ROOT = candidates[0][0] if candidates else EXPECTED
    print(f'\n[WARN] Expected path empty or missing — auto-selected: {ROOT}')

print(f'\nDataset ROOT : {ROOT}')
print(f'ROOT exists  : {ROOT.exists()}')
print(f'Images found : {_count_images(ROOT)}')
print(f'Kaggle GPU   : {os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "unknown")}')

## 1. Folder Structure — Full Tree

In [ ]:
# ── Full recursive tree ───────────────────────────────────────────────────────
def print_tree(path: Path, prefix='', max_depth=6, depth=0):
    if depth > max_depth:
        return
    entries = sorted(path.iterdir()) if path.is_dir() else []
    dirs  = [e for e in entries if e.is_dir()]
    files = [e for e in entries if e.is_file()]
    # summarise large file lists
    shown_files = files[:5]
    for f in shown_files:
        print(prefix + '├── ' + f.name + f'  ({f.stat().st_size/1024:.1f} KB)')
    if len(files) > 5:
        print(prefix + f'└── ... ({len(files)-5} more files)')
    for d in dirs:
        n_sub = sum(1 for _ in d.rglob('*') if _.is_file())
        print(prefix + '├── [DIR] ' + d.name + f'  ({n_sub} files)')
        print_tree(d, prefix + '│   ', max_depth, depth+1)

print(str(ROOT))
print_tree(ROOT)

In [ ]:
# ── Inventory every file ──────────────────────────────────────────────────────
all_files = list(ROOT.rglob('*'))
rows = [
    {
        'path': str(f),
        'name': f.name,
        'suffix': f.suffix.lower(),
        'size_kb': f.stat().st_size / 1024,
        'parent': str(f.parent.relative_to(ROOT))
    }
    for f in all_files if f.is_file()
]
file_df = pd.DataFrame(rows) if rows else pd.DataFrame(
    columns=['path', 'name', 'suffix', 'size_kb', 'parent'])

print('Total files:', len(file_df))

if file_df.empty:
    print('\n[ERROR] No files under ROOT. Listing all of /kaggle/input/ for diagnosis:')
    for p in sorted(Path('/kaggle/input').rglob('*'))[:80]:
        marker = '[DIR]' if p.is_dir() else f'[{p.stat().st_size//1024}KB]'
        print(f'  {marker}  {p}')
    raise RuntimeError(
        f'Dataset not found. ROOT={ROOT} contains 0 files. '
        'Ensure the dataset is attached to this kernel.'
    )

print('\nBy extension:')
print(file_df['suffix'].value_counts().to_string())
print('\nBy top-level folder:')
file_df['top_folder'] = file_df['path'].apply(
    lambda p: Path(p).relative_to(ROOT).parts[0]
    if len(Path(p).relative_to(ROOT).parts) > 1 else '__root__'
)
print(file_df.groupby('top_folder')['name'].count().sort_values(ascending=False).to_string())

## 2. Annotation Files — Parse Every Format

In [ ]:
# ── Auto-detect annotation format ────────────────────────────────────────────
ann_paths = list(ROOT.rglob('*.json')) + list(ROOT.rglob('*.csv')) + \
            list(ROOT.rglob('*.txt')) + list(ROOT.rglob('*.xml'))
print('Annotation / metadata files found:')
for p in ann_paths:
    size = p.stat().st_size
    print(f'  {p.relative_to(ROOT)}  ({size/1024:.1f} KB)')

In [ ]:
# ── Parse each JSON (COCO or custom) ─────────────────────────────────────────
def inspect_json(path):
    with open(path) as f:
        d = json.load(f)
    print(f'\n=== {path.relative_to(ROOT)} ===')
    if isinstance(d, dict):
        for k, v in d.items():
            if isinstance(v, list):
                print(f'  [{k}] → {len(v)} items')
                if v:
                    print(f'     first keys: {list(v[0].keys()) if isinstance(v[0], dict) else v[:3]}')
            else:
                print(f'  {k}: {str(v)[:120]}')
    elif isinstance(d, list):
        print(f'  List of {len(d)} items')
        if d:
            print(f'  First item keys: {list(d[0].keys()) if isinstance(d[0], dict) else d[0]}')
    return d

parsed_jsons = {}
for p in ROOT.rglob('*.json'):
    parsed_jsons[str(p.relative_to(ROOT))] = inspect_json(p)

In [ ]:
# ── Parse CSV files ───────────────────────────────────────────────────────────
csv_dfs = {}
for p in ROOT.rglob('*.csv'):
    df = pd.read_csv(p)
    csv_dfs[str(p.relative_to(ROOT))] = df
    print(f'\n=== {p.relative_to(ROOT)} ===')
    print(f'  Shape: {df.shape}')
    print(f'  Columns: {list(df.columns)}')
    print(df.head(3).to_string())
    print('\n  Dtypes:')
    print(df.dtypes.to_string())
    print('\n  Null counts:')
    print(df.isnull().sum().to_string())

In [ ]:
# ── Parse TXT files (YOLO format check) ──────────────────────────────────────
txt_files = list(ROOT.rglob('*.txt'))
print(f'TXT files: {len(txt_files)}')
for p in txt_files[:10]:
    lines = p.read_text().strip().split('\n')
    print(f'  {p.relative_to(ROOT)}: {len(lines)} lines | sample: {lines[0][:120]}')

## 3. Build Master Annotation DataFrame

In [ ]:
# ── Strategy: try COCO JSON first, then CSV, then filename parsing ────────────
records = []

# --- COCO JSON path ----------------------------------------------------------
def load_coco(json_data, json_path):
    """Extract image-level records from a COCO-format JSON."""
    if not isinstance(json_data, dict):
        return []
    if 'images' not in json_data:
        return []
    img_map = {img['id']: img for img in json_data.get('images', [])}
    ann_map = defaultdict(list)
    for ann in json_data.get('annotations', []):
        ann_map[ann['image_id']].append(ann)
    rows = []
    for img_id, img in img_map.items():
        anns = ann_map.get(img_id, [])
        row = {
            'image_id': img_id,
            'file_name': img.get('file_name', ''),
            'width': img.get('width', np.nan),
            'height': img.get('height', np.nan),
            'weight_kg': img.get('weight', img.get('weight_kg', np.nan)),
            'age': img.get('age', np.nan),
            'sex': img.get('sex', img.get('gender', '')),
            'breed': img.get('breed', ''),
            'viewpoint': img.get('viewpoint', img.get('view', '')),
            'n_annotations': len(anns),
            'has_segmentation': any('segmentation' in a and a['segmentation'] for a in anns),
            'has_keypoints': any('keypoints' in a for a in anns),
            'source_json': str(Path(json_path).relative_to(ROOT)),
        }
        # pull weight from annotation if not in image dict
        if np.isnan(row['weight_kg']) and anns:
            row['weight_kg'] = anns[0].get('weight', anns[0].get('weight_kg', np.nan))
        rows.append(row)
    return rows

for key, data in parsed_jsons.items():
    rows = load_coco(data, ROOT / key)
    records.extend(rows)

# --- CSV path ----------------------------------------------------------------
for key, df in csv_dfs.items():
    # Normalize column names
    col_map = {}
    for c in df.columns:
        cl = c.lower().strip()
        if 'weight' in cl:   col_map[c] = 'weight_kg'
        elif 'age' in cl:    col_map[c] = 'age'
        elif 'sex' in cl or 'gender' in cl: col_map[c] = 'sex'
        elif 'breed' in cl:  col_map[c] = 'breed'
        elif 'file' in cl or 'image' in cl or 'img' in cl: col_map[c] = 'file_name'
        elif 'view' in cl:   col_map[c] = 'viewpoint'
    dfr = df.rename(columns=col_map)
    dfr['source_csv'] = key
    records.extend(dfr.to_dict('records'))

print(f'Total raw records assembled: {len(records)}')

master = pd.DataFrame(records)
print('\nMaster columns:', list(master.columns))
print(master.head())

In [ ]:
# ── Fallback: parse weight from filename if annotations are missing ────────────
# Many cattle datasets embed weight in filename like: cow_320kg_001.jpg
img_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}
img_files = [f for f in all_files if f.is_file() and f.suffix.lower() in img_extensions]
print(f'Total image files found: {len(img_files)}')

import re
def parse_weight_from_filename(name: str):
    """Try to extract weight in kg from filename."""
    name = name.lower()
    patterns = [
        r'(\d{2,4})\s*kg',
        r'w(\d{2,4})',
        r'weight[_\-]?(\d{2,4})',
        r'_(\d{2,4})_',
    ]
    for pat in patterns:
        m = re.search(pat, name)
        if m:
            val = int(m.group(1))
            if 50 <= val <= 1500:
                return val
    return np.nan

filename_records = []
for f in img_files:
    w = parse_weight_from_filename(f.stem)
    filename_records.append({
        'file_name': f.name,
        'full_path': str(f),
        'folder': str(f.parent.relative_to(ROOT)),
        'weight_from_name': w,
    })

img_df = pd.DataFrame(filename_records)
n_with_weight_in_name = img_df['weight_from_name'].notna().sum()
print(f'Images with weight in filename: {n_with_weight_in_name} / {len(img_df)}')
print('\nSample filenames:')
print(img_df['file_name'].head(20).to_string())

In [ ]:
# ── Merge annotation records with image list ───────────────────────────────────
# Build a unified per-image dataframe
if len(master) > 0 and 'file_name' in master.columns:
    master['file_name_base'] = master['file_name'].apply(lambda x: Path(str(x)).name if pd.notna(x) else '')
    img_df['file_name_base'] = img_df['file_name'].apply(lambda x: Path(str(x)).name)
    merged = img_df.merge(master.drop_duplicates('file_name_base'),
                          on='file_name_base', how='left', suffixes=('_img', '_ann'))
else:
    merged = img_df.copy()
    merged['weight_kg'] = merged['weight_from_name']

# Coalesce weight columns
if 'weight_kg' not in merged.columns:
    merged['weight_kg'] = np.nan
merged['weight_kg'] = merged['weight_kg'].fillna(merged.get('weight_from_name', np.nan))

print('Unified dataset shape:', merged.shape)
print('weight_kg available:', merged['weight_kg'].notna().sum(), '/', len(merged))
print('\nColumn nulls:')
print(merged.isnull().sum().sort_values(ascending=False).head(20).to_string())

## 4. Image Resolution Analysis

In [ ]:
# ── Sample resolution from up to 1000 images ─────────────────────────────────
sample_paths = img_files[:1000] if len(img_files) > 1000 else img_files
res_records = []
for p in tqdm(sample_paths, desc='Reading resolutions'):
    try:
        with Image.open(p) as im:
            w, h = im.size
            mode = im.mode
        res_records.append({'path': str(p), 'width': w, 'height': h,
                             'mode': mode, 'aspect': w/h})
    except Exception as e:
        res_records.append({'path': str(p), 'width': 0, 'height': 0,
                             'mode': 'ERROR', 'aspect': np.nan, 'error': str(e)})

res_df = pd.DataFrame(res_records)
print('Resolution summary:')
print(res_df[['width','height','aspect']].describe().round(1).to_string())
print('\nModes:', res_df['mode'].value_counts().to_string())
print('\nCorrupted:', (res_df['mode'] == 'ERROR').sum())
print('\nResolution pairs (top 20):')
print(res_df.groupby(['width','height']).size().sort_values(ascending=False).head(20).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(res_df['width'].clip(0, 5000), bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Image Width Distribution'); axes[0].set_xlabel('pixels')

axes[1].hist(res_df['height'].clip(0, 5000), bins=40, color='coral', edgecolor='white')
axes[1].set_title('Image Height Distribution'); axes[1].set_xlabel('pixels')

axes[2].hist(res_df['aspect'].dropna().clip(0.2, 5), bins=40, color='mediumseagreen', edgecolor='white')
axes[2].set_title('Aspect Ratio (W/H) Distribution'); axes[2].set_xlabel('W/H ratio')
axes[2].axvline(1.0, color='red', linestyle='--', label='square')
axes[2].legend()

plt.tight_layout()
plt.savefig('resolution_distributions.png', dpi=120)
plt.show()
print('Landscape images (W>H):', (res_df['aspect'] > 1).sum())
print('Portrait images (H>W):', (res_df['aspect'] < 1).sum())

## 5. Weight Distribution Analysis

In [ ]:
weights = merged['weight_kg'].dropna()
print(f'Weight records: {len(weights)}')
print(f'Min: {weights.min():.1f} kg')
print(f'Max: {weights.max():.1f} kg')
print(f'Mean: {weights.mean():.1f} kg')
print(f'Median: {weights.median():.1f} kg')
print(f'Std: {weights.std():.1f} kg')
print(f'CV (std/mean): {weights.std()/weights.mean():.3f}')
print(f'Skewness: {weights.skew():.3f}')
print(f'Kurtosis: {weights.kurtosis():.3f}')
print('\nPercentiles:')
for p in [5, 10, 25, 50, 75, 90, 95, 99]:
    print(f'  P{p}: {np.percentile(weights, p):.1f} kg')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Histogram
axes[0,0].hist(weights, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0,0].axvline(weights.mean(), color='red', linestyle='--', label=f'Mean={weights.mean():.0f}')
axes[0,0].axvline(weights.median(), color='orange', linestyle='--', label=f'Median={weights.median():.0f}')
axes[0,0].set_title('Weight Distribution (kg)'); axes[0,0].legend()

# Log-scale histogram
axes[0,1].hist(np.log1p(weights), bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[0,1].set_title('Log(1+Weight) Distribution')

# Boxplot
axes[1,0].boxplot(weights, vert=False, patch_artist=True,
                  boxprops=dict(facecolor='lightblue'))
axes[1,0].set_title('Weight Boxplot (outlier detection)')
axes[1,0].set_xlabel('kg')

# QQ plot
from scipy.stats import probplot
probplot(weights, plot=axes[1,1])
axes[1,1].set_title('Normal Q-Q Plot of Weights')

plt.tight_layout()
plt.savefig('weight_distributions.png', dpi=120)
plt.show()

# IQR-based outliers
Q1, Q3 = weights.quantile(0.25), weights.quantile(0.75)
IQR = Q3 - Q1
outliers = weights[(weights < Q1 - 3*IQR) | (weights > Q3 + 3*IQR)]
print(f'\nHard outliers (3×IQR): {len(outliers)} samples')
print(outliers.sort_values().to_string())

## 6. Age, Sex, Breed Distribution

In [ ]:
for col in ['age', 'sex', 'breed', 'viewpoint']:
    if col in merged.columns:
        vals = merged[col].dropna()
        print(f'\n--- {col} ---')
        print(f'  Non-null: {len(vals)} / {len(merged)}')
        if vals.dtype == object or col in ['sex','breed','viewpoint']:
            print(vals.value_counts().head(20).to_string())
        else:
            print(f'  Min={vals.min():.1f}  Max={vals.max():.1f}  Mean={vals.mean():.1f}  Std={vals.std():.1f}')
    else:
        print(f'\n--- {col} --- NOT PRESENT')

In [ ]:
# ── Weight vs categorical variables ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
cat_cols = ['sex', 'breed', 'viewpoint']
for ax, col in zip(axes, cat_cols):
    if col in merged.columns and merged[col].notna().sum() > 0:
        sub = merged[['weight_kg', col]].dropna()
        groups = [sub.loc[sub[col]==v, 'weight_kg'].values
                  for v in sub[col].unique()]
        labels = [str(v) for v in sub[col].unique()]
        ax.boxplot(groups, labels=labels, patch_artist=True)
        ax.set_title(f'Weight vs {col}')
        ax.set_ylabel('Weight (kg)')
        ax.tick_params(axis='x', rotation=45)
    else:
        ax.text(0.5, 0.5, f'{col} not available', ha='center', va='center',
                transform=ax.transAxes, fontsize=14, color='gray')
        ax.set_title(f'Weight vs {col} (N/A)')
plt.tight_layout()
plt.savefig('weight_vs_categories.png', dpi=120)
plt.show()

In [ ]:
# ── Age vs Weight scatter ─────────────────────────────────────────────────────
if 'age' in merged.columns and merged['age'].notna().sum() > 10:
    sub = merged[['age','weight_kg']].dropna()
    r, p = stats.pearsonr(sub['age'], sub['weight_kg'])
    print(f'Age vs Weight: Pearson r={r:.3f}, p={p:.4f}')
    plt.figure(figsize=(8,5))
    plt.scatter(sub['age'], sub['weight_kg'], alpha=0.4, s=15)
    m, b = np.polyfit(sub['age'], sub['weight_kg'], 1)
    xs = np.linspace(sub['age'].min(), sub['age'].max(), 100)
    plt.plot(xs, m*xs+b, 'r--', label=f'y={m:.1f}x+{b:.0f}')
    plt.xlabel('Age'); plt.ylabel('Weight (kg)')
    plt.title(f'Age vs Weight  (r={r:.3f})')
    plt.legend(); plt.savefig('age_vs_weight.png', dpi=120); plt.show()
else:
    print('Age column not available or insufficient data')

## 7. Image Quality — Blur, Contrast, Brightness

In [ ]:
# ── Laplacian variance for blur detection ─────────────────────────────────────
N_QUALITY = min(500, len(img_files))
quality_records = []
for p in tqdm(img_files[:N_QUALITY], desc='Image quality scan'):
    try:
        img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if img is None:
            quality_records.append({'path': str(p), 'blur': np.nan,
                                     'brightness': np.nan, 'contrast': np.nan})
            continue
        lap_var = cv2.Laplacian(img, cv2.CV_64F).var()
        brightness = img.mean()
        contrast = img.std()
        quality_records.append({'path': str(p),
                                  'blur_score': lap_var,
                                  'brightness': brightness,
                                  'contrast': contrast})
    except Exception as e:
        quality_records.append({'path': str(p), 'blur_score': np.nan,
                                  'brightness': np.nan, 'contrast': np.nan,
                                  'error': str(e)})

qual_df = pd.DataFrame(quality_records)
print('Quality stats:')
print(qual_df[['blur_score','brightness','contrast']].describe().round(2).to_string())
BLUR_THRESHOLD = 100  # Laplacian variance below this → likely blurry
print(f'\nBlurry images (lap_var < {BLUR_THRESHOLD}): {(qual_df["blur_score"] < BLUR_THRESHOLD).sum()}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col, color in zip(axes,
    ['blur_score','brightness','contrast'],
    ['steelblue','coral','mediumseagreen']):
    data = qual_df[col].dropna()
    ax.hist(data.clip(data.quantile(0.01), data.quantile(0.99)),
            bins=40, color=color, edgecolor='white')
    ax.set_title(f'{col} Distribution')
    ax.set_xlabel(col)
plt.tight_layout()
plt.savefig('image_quality.png', dpi=120)
plt.show()

## 8. Duplicate & Near-Duplicate Detection

In [ ]:
# ── MD5 exact duplicates ───────────────────────────────────────────────────────
N_DUP = min(2000, len(img_files))
hash_records = []
for p in tqdm(img_files[:N_DUP], desc='Hashing images'):
    try:
        md5 = hashlib.md5(p.read_bytes()).hexdigest()
        hash_records.append({'path': str(p), 'md5': md5})
    except:
        pass

hash_df = pd.DataFrame(hash_records)
dup_md5 = hash_df[hash_df.duplicated('md5', keep=False)]
print(f'Exact duplicate images: {len(dup_md5)} ({len(dup_md5)/max(len(hash_df),1)*100:.1f}%)')
print(dup_md5.groupby('md5')['path'].apply(list).head(5).to_string())

In [ ]:
# ── Perceptual hash near-duplicates ───────────────────────────────────────────
N_PHASH = min(500, len(img_files))
phash_records = []
for p in tqdm(img_files[:N_PHASH], desc='Perceptual hashing'):
    try:
        with Image.open(p) as im:
            ph = imagehash.phash(im)
        phash_records.append({'path': str(p), 'phash': ph})
    except:
        pass

# Count near-dups (hamming distance <= 8 out of 64 bits)
near_dup_pairs = []
for i in range(len(phash_records)):
    for j in range(i+1, len(phash_records)):
        dist = phash_records[i]['phash'] - phash_records[j]['phash']
        if dist <= 8:
            near_dup_pairs.append((phash_records[i]['path'], phash_records[j]['path'], dist))

print(f'Near-duplicate pairs (hamming ≤ 8): {len(near_dup_pairs)}')
for p1, p2, d in near_dup_pairs[:5]:
    print(f'  dist={d}: {Path(p1).name}  ↔  {Path(p2).name}')

## 9. Mask & Segmentation Quality

In [ ]:
# ── Find mask files ───────────────────────────────────────────────────────────
mask_extensions = {'.png', '.jpg', '.jpeg'}
mask_dirs = [d for d in ROOT.rglob('*') if d.is_dir() and
             any(kw in d.name.lower() for kw in ['mask','seg','label','annot'])]
print('Potential mask directories:')
for d in mask_dirs:
    files_in = list(d.rglob('*'))
    imgs_in = [f for f in files_in if f.is_file() and f.suffix.lower() in mask_extensions]
    print(f'  {d.relative_to(ROOT)}: {len(imgs_in)} image files')

In [ ]:
# ── Measure mask quality metrics ──────────────────────────────────────────────
mask_files = []
for d in mask_dirs:
    mask_files.extend([f for f in d.rglob('*')
                        if f.is_file() and f.suffix.lower() in mask_extensions])

# Also look for binary masks (pixel values 0 or 255)
if not mask_files:
    # Some datasets store masks in same folder with '_mask' suffix
    for f in img_files:
        candidates = [
            f.parent / (f.stem + '_mask' + f.suffix),
            f.parent / 'masks' / f.name,
            f.parent / 'Masks' / f.name,
        ]
        for c in candidates:
            if c.exists():
                mask_files.append(c)
                break

print(f'Mask files found: {len(mask_files)}')

# Analyze mask quality
N_MASK = min(200, len(mask_files))
mask_records = []
for p in tqdm(mask_files[:N_MASK], desc='Analyzing masks'):
    try:
        mask = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            continue
        total_px = mask.size
        fg_px = (mask > 127).sum()
        fill_ratio = fg_px / total_px
        # check binary
        unique_vals = np.unique(mask)
        is_binary = len(unique_vals) <= 3  # 0, maybe 128, 255
        # contour count
        _, thresh = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        n_contours = len(contours)
        mask_records.append({
            'path': str(p),
            'width': mask.shape[1],
            'height': mask.shape[0],
            'fill_ratio': fill_ratio,
            'is_binary': is_binary,
            'n_contours': n_contours,
            'n_unique_vals': len(unique_vals),
        })
    except Exception as e:
        pass

if mask_records:
    mask_df = pd.DataFrame(mask_records)
    print('\nMask quality summary:')
    print(mask_df[['fill_ratio','n_contours']].describe().round(3).to_string())
    print(f'Binary masks: {mask_df["is_binary"].mean()*100:.1f}%')
    print(f'Multi-object masks (n_contours>1): {(mask_df["n_contours"]>1).sum()}')
else:
    print('No mask files found or accessible.')
    mask_df = pd.DataFrame()

In [ ]:
# ── Visualize sample mask quality ─────────────────────────────────────────────
if len(mask_files) > 0:
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    for ax, p in zip(axes.flat, mask_files[:12]):
        img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if img is not None:
            ax.imshow(img, cmap='gray')
            ax.set_title(p.name[:20], fontsize=8)
        ax.axis('off')
    plt.suptitle('Sample Masks', fontsize=14)
    plt.tight_layout()
    plt.savefig('sample_masks.png', dpi=120)
    plt.show()

## 10. Keypoint Analysis

In [ ]:
# ── Extract keypoints from COCO annotations ────────────────────────────────────
all_keypoints = []
kp_names = []
for key, data in parsed_jsons.items():
    if not isinstance(data, dict):
        continue
    cats = data.get('categories', [])
    for cat in cats:
        if 'keypoints' in cat:
            kp_names = cat['keypoints']
            print(f'Keypoint names in {key}: {kp_names}')
    for ann in data.get('annotations', []):
        kps = ann.get('keypoints', [])
        if kps:
            all_keypoints.append({'image_id': ann.get('image_id'),
                                   'n_keypoints': len(kps)//3,
                                   'keypoints': kps})

print(f'\nAnnotations with keypoints: {len(all_keypoints)}')
if all_keypoints:
    kp_df = pd.DataFrame(all_keypoints)
    # Visibility stats
    # COCO keypoints: [x, y, v] where v=0 not labeled, v=1 labeled not visible, v=2 labeled visible
    vis_counts = []
    for row in all_keypoints:
        kps = row['keypoints']
        vs = [kps[i] for i in range(2, len(kps), 3)]
        vis_counts.append({'n_visible': sum(v==2 for v in vs),
                            'n_labeled': sum(v>0 for v in vs),
                            'n_total': len(vs)})
    vis_df = pd.DataFrame(vis_counts)
    print('\nKeypoint visibility summary:')
    print(vis_df.describe().round(2).to_string())

## 11. Body Measurement Feature Engineering

In [ ]:
# ── Extract geometric features from binary masks ──────────────────────────────
def extract_mask_features(mask_path):
    """Extract body measurement proxies from a binary segmentation mask."""
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None
    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    H, W = binary.shape
    total_px = H * W

    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    cnt = max(contours, key=cv2.contourArea)

    area = cv2.contourArea(cnt)
    perimeter = cv2.arcLength(cnt, True)
    hull = cv2.convexHull(cnt)
    hull_area = cv2.contourArea(hull)
    solidity = area / (hull_area + 1e-6)

    x, y, w, h = cv2.boundingRect(cnt)
    aspect_ratio = w / (h + 1e-6)
    extent = area / (w * h + 1e-6)
    fill_ratio = area / total_px

    # Fit ellipse
    el_area, el_axes, el_angle = np.nan, (np.nan, np.nan), np.nan
    if len(cnt) >= 5:
        ellipse = cv2.fitEllipse(cnt)
        el_center, el_axes, el_angle = ellipse
        el_area = np.pi * el_axes[0]/2 * el_axes[1]/2

    # Moments — centroid
    M = cv2.moments(cnt)
    cx = M['m10'] / (M['m00'] + 1e-6)
    cy = M['m01'] / (M['m00'] + 1e-6)
    # Normalize centroid
    cx_norm = cx / W
    cy_norm = cy / H

    # Bbox proportions
    bbox_w_norm = w / W
    bbox_h_norm = h / H

    # Skeleton length approximation (thinning)
    roi = binary[y:y+h, x:x+w]
    roi_small = cv2.resize(roi, (128, 128)) > 127
    from skimage.morphology import skeletonize
    skel = skeletonize(roi_small)
    skel_length = skel.sum()

    return {
        'mask_area_px': area,
        'mask_area_ratio': fill_ratio,
        'perimeter': perimeter,
        'hull_area': hull_area,
        'solidity': solidity,
        'bbox_w': w, 'bbox_h': h,
        'bbox_w_norm': bbox_w_norm,
        'bbox_h_norm': bbox_h_norm,
        'aspect_ratio_bbox': aspect_ratio,
        'extent': extent,
        'cx_norm': cx_norm, 'cy_norm': cy_norm,
        'ellipse_major': el_axes[0] if not np.isnan(el_area) else np.nan,
        'ellipse_minor': el_axes[1] if not np.isnan(el_area) else np.nan,
        'ellipse_angle': el_angle,
        'skel_length': skel_length,
        'compactness': (4 * np.pi * area) / (perimeter**2 + 1e-6),
    }

# Compute for all available masks
feat_records = []
N_FEAT = min(300, len(mask_files))
for p in tqdm(mask_files[:N_FEAT], desc='Extracting mask features'):
    feats = extract_mask_features(p)
    if feats:
        feats['mask_path'] = str(p)
        feats['mask_name'] = p.name
        feat_records.append(feats)

feat_df = pd.DataFrame(feat_records)
print(f'Feature records: {len(feat_df)}')
print(feat_df.describe().round(3).to_string())

In [ ]:
# ── Correlate features with weight ───────────────────────────────────────────
# Merge feat_df with weight labels
feat_df['base_name'] = feat_df['mask_name'].apply(
    lambda n: re.sub(r'_mask|_seg|_label', '', Path(n).stem, flags=re.IGNORECASE)
)
if 'file_name' in merged.columns:
    merged['base_name'] = merged['file_name'].apply(
        lambda n: Path(str(n)).stem if pd.notna(n) else ''
    )
    feat_merged = feat_df.merge(merged[['base_name','weight_kg']].dropna(),
                                 on='base_name', how='inner')
else:
    feat_merged = pd.DataFrame()

numeric_feat_cols = ['mask_area_ratio','aspect_ratio_bbox','bbox_w_norm','bbox_h_norm',
                      'solidity','compactness','extent','skel_length',
                      'ellipse_major','ellipse_minor']

if len(feat_merged) > 10:
    corrs = {}
    for c in numeric_feat_cols:
        if c in feat_merged.columns:
            sub = feat_merged[[c,'weight_kg']].dropna()
            if len(sub) > 5:
                r, p = stats.pearsonr(sub[c], sub['weight_kg'])
                corrs[c] = {'pearson_r': r, 'p_value': p, 'n': len(sub)}
    corr_df = pd.DataFrame(corrs).T.sort_values('pearson_r', key=abs, ascending=False)
    print('\nFeature → Weight Correlations:')
    print(corr_df.round(4).to_string())
else:
    print('Not enough mask+weight pairs to correlate — mask labels may be separate from weight CSV.')
    print('Will proceed with image-only feature analysis.')

## 12. Sample Image Grid — Visual Inspection

In [ ]:
# ── Show 20 random samples ────────────────────────────────────────────────────
import random
random.seed(42)
sample_imgs = random.sample(img_files, min(20, len(img_files)))

fig, axes = plt.subplots(4, 5, figsize=(20, 16))
for ax, p in zip(axes.flat, sample_imgs):
    try:
        img = Image.open(p).convert('RGB')
        img.thumbnail((400, 400))
        ax.imshow(img)
        # Try to get weight
        name_base = p.stem
        w_val = np.nan
        if 'file_name' in merged.columns:
            row = merged[merged['file_name'].str.contains(p.name, na=False, regex=False)]
            if len(row):
                w_val = row['weight_kg'].values[0]
        title = f'{p.name[:18]}'
        if not np.isnan(w_val):
            title += f'\n{w_val:.0f}kg'
        ax.set_title(title, fontsize=7)
    except Exception as e:
        ax.text(0.5, 0.5, f'Error: {e}', ha='center', fontsize=7,
                transform=ax.transAxes)
    ax.axis('off')

plt.suptitle('Random Sample Images', fontsize=16)
plt.tight_layout()
plt.savefig('sample_images.png', dpi=120)
plt.show()

In [ ]:
# ── Show heaviest vs lightest cows ────────────────────────────────────────────
if len(merged) > 0 and 'weight_kg' in merged.columns:
    w_sorted = merged.dropna(subset=['weight_kg','full_path']).sort_values('weight_kg')
    extremes = pd.concat([w_sorted.head(5), w_sorted.tail(5)])
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    for ax, (_, row) in zip(axes.flat, extremes.iterrows()):
        try:
            img = Image.open(row['full_path']).convert('RGB')
            img.thumbnail((300, 300))
            ax.imshow(img)
            ax.set_title(f"{row['weight_kg']:.0f} kg", fontsize=11,
                         color='blue' if row['weight_kg'] < w_sorted['weight_kg'].median() else 'red')
        except:
            ax.text(0.5, 0.5, 'Load error', ha='center', transform=ax.transAxes)
        ax.axis('off')
    axes[0, 0].set_ylabel('LIGHTEST →', fontsize=12, color='blue')
    axes[1, 0].set_ylabel('← HEAVIEST', fontsize=12, color='red')
    plt.suptitle('Lightest (top row) vs Heaviest (bottom row)', fontsize=14)
    plt.tight_layout()
    plt.savefig('weight_extremes.png', dpi=120)
    plt.show()

## 13. MAPE Achievability Analysis

In [ ]:
print('=' * 70)
print('MAPE ACHIEVABILITY SCIENTIFIC ANALYSIS')
print('=' * 70)

if len(weights) > 0:
    cv = weights.std() / weights.mean()
    n = len(weights)
    weight_range = weights.max() - weights.min()

    print(f'\nDataset size (with weight labels): {n}')
    print(f'Weight range: {weights.min():.0f} – {weights.max():.0f} kg')
    print(f'CV (std/mean): {cv:.3f}')

    # Theoretical lower bound: aleatoric noise from imaging
    # For livestock weight estimation from images, published results:
    # - With depth sensors: 2-4% MAPE
    # - With RGB only + segmentation: 5-10% MAPE
    # - With RGB only (CNN): 8-15% MAPE
    # Key factors:

    print(f'''
THEORETICAL ANALYSIS:

1. INFORMATION CONTENT OF A 2D IMAGE
   A side-view 2D image captures:
   - Body length, height (lateral projection): HIGH correlation with weight
   - Body depth (front-to-back): NOT directly visible — reduces accuracy
   - Abdominal fill (pregnancy, rumen fill): NOT captured — adds noise
   - Breed-specific density differences: NOT visible — requires metadata

   Estimated aleatoric noise floor: ~4-5% MAPE
   (irreducible given single 2D viewpoint)

2. DATASET SIZE EFFECT (n={n})
   - n < 500:  model likely overfits → MAPE > 12%
   - n = 500–2000: reasonable → MAPE 8-12%
   - n > 2000: good → MAPE 6-10%
   - n > 5000: strong → MAPE 5-8%
   - n > 10000: excellent → MAPE 4-7%

3. LABEL NOISE
   Cattle are typically weighed with scales having ±2-5 kg error.
   For a 300 kg cow: ±5 kg = ±1.7%  (acceptable)
   For a 150 kg cow: ±5 kg = ±3.3%  (significant)

4. BODY CONDITION VARIABILITY
   Two cows of same weight can look very different due to:
   - Pregnancy stage
   - Rumen fill (how recently fed)
   - Breed
   - Frame score
   This typically adds 3-6% prediction noise.

VERDICT:
   With n={n} samples and 2D RGB only:
   - 5% MAPE: VERY CHALLENGING — requires segmentation + keypoints + metadata
   - 6% MAPE: ACHIEVABLE — with multi-input model (image + mask + features)
   - 7% MAPE: LIKELY — with strong CNN + mask features
   - 8% MAPE: EASILY ACHIEVABLE — baseline CNN
''')

## 14. Comprehensive Summary Report

In [ ]:
print('=' * 70)
print('COMPLETE DATASET AUDIT REPORT')
print('=' * 70)
print(f'\nTotal image files: {len(img_files)}')
print(f'Total annotation records: {len(merged)}')
print(f'Records with weight labels: {merged["weight_kg"].notna().sum() if "weight_kg" in merged.columns else 0}')
print(f'Mask files available: {len(mask_files)}')
print(f'Keypoint annotations: {len(all_keypoints)}')
print()

# Missing values
print('MISSING VALUES:')
for col in ['weight_kg','age','sex','breed','viewpoint']:
    if col in merged.columns:
        n_miss = merged[col].isna().sum()
        pct = n_miss / len(merged) * 100
        print(f'  {col}: {n_miss} missing ({pct:.1f}%)')

print()
print('IMAGE QUALITY:')
if 'blur_score' in qual_df.columns:
    print(f'  Blurry images: {(qual_df["blur_score"]<100).sum()} / {len(qual_df)}')
print(f'  Exact duplicates: {len(dup_md5)}')
print(f'  Near duplicates: {len(near_dup_pairs)}')

print()
print('RESOLUTION:')
if len(res_df) > 0:
    print(f'  Most common: {res_df.groupby(["width","height"]).size().idxmax()}')
    print(f'  Range W: {res_df["width"].min()}-{res_df["width"].max()} px')
    print(f'  Range H: {res_df["height"].min()}-{res_df["height"].max()} px')

print()
print('WEIGHT LABELS:')
if 'weight_kg' in merged.columns and merged['weight_kg'].notna().sum() > 0:
    w = merged['weight_kg'].dropna()
    print(f'  Range: {w.min():.0f} – {w.max():.0f} kg')
    print(f'  Mean ± Std: {w.mean():.1f} ± {w.std():.1f} kg')
    print(f'  Outliers (3×IQR): {len(outliers)}')

print()
print('RECOMMENDED NEXT STEPS:')
print('  1. Proceed to Step 2: Feature Engineering')
print('  2. Build baseline CNN regression')
print('  3. Add segmentation branch')
print('  4. Add keypoint / measurement branch')
print('  5. Ensemble and distill to TFLite')